In [ ]:
# ==============================================================================
# STEP 1: SETUP & LIBRARIES
# ==============================================================================
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# ตั้งค่าธีมกราฟ
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# ==============================================================================
# STEP 2: DATA LOADING (Wave 1 Baseline)
# ==============================================================================
# ใช้ convert_categoricals=False เพื่อป้องกัน Error จาก Duplicate Labels ใน Metadata
file_2007 = 'wave-1-shocksclean.dta'
df_07 = pd.read_stata(file_2007, convert_categoricals=False)

print(f"Loaded Wave 1 (2007) Baseline successfully: {len(df_07)} rows")

# ==============================================================================
# STEP 3: DATA CLEANING (Financial & Missing Values)
# ==============================================================================
def clean_wave1_baseline(df):
    # 1. จัดการค่าว่างมาตรฐาน Stata (-9, -99)
    df = df.replace([-9, -99, -9.0, -99.0], np.nan)
    
    # 2. จัดการมูลค่าความเสียหาย (_x31005)
    # แปลง 'none' เป็น 0 และทำให้เป็นตัวเลข
    if '_x31005' in df.columns:
        df['_x31005'] = df['_x31005'].astype(str).str.replace('none', '0', case=False).str.strip()
        df['_x31005'] = pd.to_numeric(df['_x31005'], errors='coerce').fillna(0)
    
    return df

df_07 = clean_wave1_baseline(df_07)

# ==============================================================================
# STEP 4: COPING STRATEGY HARMONIZATION (สร้างมิติ 0/1)
# ==============================================================================
# Mapping รหัสรับมือปี 2007 ให้เข้าหมวดหมู่มาตรฐานของ Panel Project
coping_map_07 = {
    11: 'sold_assets', 12: 'sold_assets', 13: 'sold_assets', 14: 'sold_assets',
    15: 'used_savings', 16: 'used_insurance',
    17: 'borrowed_informal', 18: 'borrowed_informal', 19: 'borrowed_informal', 20: 'borrowed_informal',
    21: 'borrowed_formal', 22: 'borrowed_formal', 23: 'borrowed_formal', 
    28: 'gov_help', 29: 'gov_help', 30: 'relatives_help'
}

# สร้างคอลัมน์ Yes/No (Binary)
coping_cols = ['sold_assets', 'used_savings', 'used_insurance', 'borrowed_informal', 'borrowed_formal', 'gov_help']
for c in coping_cols:
    df_07[f'coping_{c}'] = 0

# ตรวจสอบจาก Coping ลำดับที่ 1, 2 และ 3 (_x31008, _x31009, _x31010)
for col in ['_x31008', '_x31009', '_x31010']:
    if col in df_07.columns:
        for code, cat_name in coping_map_07.items():
            df_07.loc[df_07[col] == code, f'coping_{cat_name}'] = 1

# ==============================================================================
# STEP 5: SHOCK GROUPING (Harmonization with Future Waves)
# ==============================================================================
# ใช้ Mapping เดียวกับ Wave อื่นๆ เพื่อความต่อเนื่องของข้อมูล Panel
shock_map_07 = {
    10: 'agricultural', 11: 'agricultural', 63: 'agricultural', 55: 'agricultural',
    1: 'demographic', 2: 'demographic', 3: 'demographic', 24: 'demographic',
    5: 'economics', 6: 'economics', 18: 'economics', 21: 'economics', 22: 'economics', 62: 'economics',
    8: 'social', 70: 'social', 77: 'economics'
}

df_07['shocks_Group'] = df_07['_x31002'].map(shock_map_07).fillna('others')
df_07['survey_year'] = 2007

# ==============================================================================
# STEP 6: EXPORT
# ==============================================================================
output_file = 'shocks_2007_cleaned_final.csv'
df_07.to_csv(output_file, index=False)

# แสดงกราฟสรุปปีฐาน
sns.countplot(data=df_07, x='shocks_Group', palette='viridis', order=df_07['shocks_Group'].value_counts().index)
plt.title('Baseline Shock Distribution (Wave 1 - 2007)')
plt.show()

print(f"✅ Baseline Wave 1 (2007) is ready: {output_file}")

In [ ]:
def run_wave1_descriptive_v2(df, year):
    # --- PRE-PROCESSING ---
    # 1. คลีนข้อมูลเงินและรวมค่าความเสียหาย (Graph 2)
    for col in ['_x31005n', '_x31006n']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col].astype(str).str.replace('none', '0'), errors='coerce').fillna(0)
    df['total_loss_sum'] = df['_x31005n'] + df['_x31006n']

    # 2. จัดการ Labels สำหรับ Recovery Status (_x31012)
    recovery_map = {
        0: "< 1 Year",
        1: "1 Year",
        2: "> 1 Year (Recovered)",
        3: "Not yet recovered"
    }
    # กรองเฉพาะค่าที่เราต้องการวิเคราะห์ (ตัด 90, 97, 98, 99 ออก)
    df['recovery_label'] = df['_x31012'].map(recovery_map)

    # 3. จัดการ Labels สำหรับ Reduce Consumption (_x31011)
    consumption_map = {1: "Yes (Reduced)", 2: "No (Did not reduce)"}
    df['consumption_label'] = df['_x31011'].map(consumption_map)

    print(f"--- Running Visual Analysis for Wave {year} ---")

    # ---------------------------------------------------------
    # GRAPH 1: Frequency of Shocks
    # ---------------------------------------------------------
    plt.figure(figsize=(10, 6))
    sns.countplot(data=df, x='shocks_Group', palette='viridis', order=df['shocks_Group'].value_counts().index)
    plt.title(f'Graph 1: Number of Shock Events by Group ({year})')
    plt.xticks(rotation=45)
    plt.show()

    # ---------------------------------------------------------
    # GRAPH 2: Average Total Loss (Summation)
    # ---------------------------------------------------------
    plt.figure(figsize=(10, 6))
    sns.barplot(data=df, x='shocks_Group', y='total_loss_sum', estimator=np.mean, palette='magma')
    plt.title(f'Graph 2: Mean Total Loss (_x31005n + _x31006n) in {year}')
    plt.ylabel('Average Value (THB)')
    plt.xticks(rotation=45)
    plt.show()

    # ---------------------------------------------------------
    # GRAPH 3: Coping Strategies
    # ---------------------------------------------------------
    plt.figure(figsize=(10, 6))
    coping_vars = [c for c in df.columns if c.startswith('coping_')]
    if coping_vars:
        df[coping_vars].sum().sort_values().plot(kind='barh', color='skyblue')
        plt.title(f'Graph 3: Coping Strategies Usage ({year})')
        plt.xlabel('Count of Households')
    plt.show()

    # ---------------------------------------------------------
    # GRAPH 4: Coping Intensity
    # ---------------------------------------------------------
    plt.figure(figsize=(10, 6))
    df['coping_count'] = df[coping_vars].sum(axis=1)
    sns.countplot(data=df, x='coping_count', palette='plasma')
    plt.title(f'Graph 4: Coping Intensity (Number of strategies per shock)')
    plt.xlabel('Strategies Combined')
    plt.show()

    # ---------------------------------------------------------
    # GRAPH 5: Recovery Status (_x31012)
    # ---------------------------------------------------------
    plt.figure(figsize=(10, 6))
    recovery_order = ["< 1 Year", "1 Year", "> 1 Year (Recovered)", "Not yet recovered"]
    sns.countplot(data=df[df['recovery_label'].notnull()], x='recovery_label', order=recovery_order, palette='Set2')
    plt.title(f'Graph 5: Time Taken to Recover from Shock (_x31012)')
    plt.ylabel('Number of Cases')
    plt.show()

    # ---------------------------------------------------------
    # GRAPH 6: Reduction in Consumption Expenditure (_x31011)
    # ---------------------------------------------------------
    plt.figure(figsize=(10, 6))
    sns.countplot(data=df[df['consumption_label'].notnull()], x='consumption_label', palette='Set1')
    plt.title(f'Graph 6: Did Household Reduce Consumption Expenditure? (_x31011)')
    plt.ylabel('Number of Households')
    plt.show()

# เรียกใช้งาน
run_wave1_descriptive_v2(df_07, 2007)